## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 3273 samples and 11582 columns.
First few rows of the data:


,overall survival days,overall survival event,samples,scan-b external id,instrument model,age at diagnosis,tumor size,lymph node group,lymph node status,er status,...,ZSWIM8,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
0,2367,0,F1,Q008818.C008840.S000215.l.r.m2.c.lib.g.k.a.t,HiSeq 2000,43,9.0,NodeNegative,NodeNegative,NaN,...,3.025197,2.918632,1.108204,3.044573,2.425306,3.244432,1.468899,6.281767,1.774107,2.437227
1,2367,0,F2,Q008769.C008792.S000250.l.r.m.c.lib.g.k.a.t,HiSeq 2000,48,14.0,1to3,NodePositive,1.0,...,3.569802,2.551831,1.678388,2.743902,1.861781,2.727427,1.924761,7.169613,1.764214,2.876100
2,2168,1,F3,Q008568.C008577.S000424.l.r.m3.c.lib.g.k.a.t,HiSeq 2000,69,27.0,4toX,NodePositive,1.0,...,5.362272,3.319094,2.443392,4.719843,1.365396,3.122333,1.619277,6.683400,2.039246,2.929346
3,2416,0,F4,Q008909.C009000.S000084.l.r.m.c.lib.g.k.a.t,HiSeq 2000,39,51.0,1to3,NodePositive,1.0,...,4.326709,2.588427,2.441704,4.157466,2.038984,3.182823,1.969433,6.666306,2.771650,2.777674
4,2389,0,F5,Q008781.C008782.S000260.l.r.m.c.lib.g.k.a.t,HiSeq 2000,73,60.0,4toX,NodePositive,1.0,...,4.573915,2.243450,1.749230,1.024820,2.032704,3.588842,2.998474,6.479006,2.696269,3.957146


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'samples'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
